# Leaseline maintenance-triage - fine-tune on a rented GPU

Trains `Qwen/Qwen2.5-1.5B-Instruct` (QLoRA, `task.type: extraction`) on the Leaseline product's
own training data (category + urgency + draft reply). See `README.md` in this folder for the
full task writeup. This notebook's own copy of `config_leaseline.yaml` (same folder) is a mirror
kept for private record-keeping - the authoritative copy lives at
`~/projects/ait-lab/ai_training/trainings/module/config_leaseline.yaml`; if they've drifted,
that one is correct.

**GPU sizing:** this base is smaller than anything else this project has trained (the 3B-scale
configs elsewhere already fit a free Colab T4). The **cheapest 16GB+ card RunPod offers**
(community-cloud RTX A4000/A5000 tier) is enough - no need for a 24GB+ card. 860 train rows,
up to 3 epochs, 2 seeds should finish in well under an hour.

**Data-safety note, read before running any cell below:** this pipeline's `FT_DATA_DIR` /
`FT_OUT_DIR` env vars point at a *shared* folder across every task unless overridden. On a
fresh pod there's nothing else there yet, so it's safe - but if you ever run this notebook on a
machine that already has other tasks' live data under the same path, override these to a
Leaseline-specific subfolder first (Cell 3 already does this - don't remove the override).

## 1. Detect platform

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    PLATFORM = "colab"
    WORKSPACE = "/content"
except ImportError:
    PLATFORM = "runpod"
    WORKSPACE = "/workspace"

print(f"platform={PLATFORM} workspace={WORKSPACE}")

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers trl peft bitsandbytes datasets accelerate pyyaml

## 3. Get the code + data onto this pod

This repo has no git remote, so code moves by scp (same convention as every other task here -
see `~/projects/ait-lab/ai_training/trainings/module/reference/RUNPOD.md`). Before running this
cell, from your **local machine**:

```bash
cd ~/projects/ait-lab/ai_training/trainings/module && bash bundle_colab.sh /tmp/module_bundle.zip
scp /tmp/module_bundle.zip root@<POD_IP>:/workspace/ft-data/
# this task's code now lives in the product's own repo, not next to module/ anymore:
scp -r ~/projects/personal/passiveincome/firstproject/private/task-leaseline-triage root@<POD_IP>:/workspace/ft-data/
# the Leaseline product's own dataset (same repo, a few folders over):
scp -r ~/projects/personal/passiveincome/firstproject/dataset root@<POD_IP>:/workspace/ft-data/leaseline-source
```

(Get `<POD_IP>` and the port from the pod's Connect panel - reassigned on every restart.)

In [ ]:
import zipfile

FT_DATA = f"{WORKSPACE}/ft-data"
MODULE_DIR = f"{FT_DATA}/module"

if not os.path.isdir(MODULE_DIR):
    zip_path = f"{FT_DATA}/module_bundle.zip"
    assert os.path.exists(zip_path), f"missing {zip_path} - scp it up first, see the cell above"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(FT_DATA)
    print(f"extracted bundle -> {MODULE_DIR}")
else:
    print(f"found existing {MODULE_DIR}, skipping extract")

assert os.path.isdir(f"{FT_DATA}/task-leaseline-triage"), "missing task-leaseline-triage - scp it up (see cell above)"
assert os.path.isdir(f"{FT_DATA}/leaseline-source"), "missing leaseline-source dataset - scp it up (see cell above)"

## 4. Isolated data/output paths - the safety override from the top of this notebook

In [ ]:
os.environ["FT_DATA_DIR"] = f"{FT_DATA}/leaseline-data"
os.environ["FT_OUT_DIR"] = f"{FT_DATA}/leaseline-runs"
os.environ["HF_HOME"] = f"{WORKSPACE}/hf-cache"  # container disk, NOT the quota-limited network volume
print("FT_DATA_DIR =", os.environ["FT_DATA_DIR"])
print("FT_OUT_DIR  =", os.environ["FT_OUT_DIR"])

## 5. Prepare data (re-run here so the pod has its own copy)

In [ ]:
%cd {MODULE_DIR}
!python ../task-leaseline-triage/prepare_data.py --config config_leaseline.yaml --source-dir {FT_DATA}/leaseline-source
!python targets.py --config config_leaseline.yaml

## 6. Human review checkpoint - look at this before training

This is real product training data, not a throwaway benchmark. Read a few rows below, then open
`config_leaseline.yaml` in this pod's file browser and flip `confirmed: false` -> `true`.

In [ ]:
import json

gold_path = f"{os.environ['FT_DATA_DIR']}/gold.jsonl"
with open(gold_path) as f:
    rows = [json.loads(l) for l in f]
print(f"{len(rows)} gold rows\n")
for r in rows[:5]:
    print("MESSAGE:", r["messages"][1]["content"])
    print("GRADED :", r["target_record"])
    print("FULL   :", r["full_target"])
    print()

## 7. Train + gate

Refuses to run until `confirmed: true` is set in the config (Step 6). This is the GPU-hours
step - everything before this was CPU-only setup.

In [ ]:
!python pipeline.py --config config_leaseline.yaml

## 8. Pull results back down

From your **local machine** (see
`~/projects/ait-lab/ai_training/trainings/module/reference/RUNPOD.md` for the S3-API fallback if
the pod becomes unreachable):

```bash
ssh root@<POD_IP> "cd /workspace/ft-data/leaseline-runs && zip -r -x '*/checkpoint-*/*' report.zip ."
scp root@<POD_IP>:/workspace/ft-data/leaseline-runs/report.zip .
unzip report.zip -d leaseline-runs-result
cat leaseline-runs-result/report.json   # the ACCEPT/REJECT verdict + mean_field_accuracy gain
```